In [2]:
import os
os.chdir("../")
%pwd

'C:\\Users\\Sayantan Nandi\\Desktop\\mlops\\developer-burnout-project'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    preprocessor_path: Path

In [4]:
from src.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.utils.common import read_yaml, create_directories

In [14]:
class ConfigurationManager:
    def __init__(self,
                 config_path=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                ):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir, config.preprocessor_path])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            preprocessor_path=config.preprocessor_path
        )

        return data_transformation_config

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
import joblib

In [12]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        create_directories([self.config.root_dir])

    def transform_and_save(self):
        data = pd.read_csv(self.config.data_path)

        # 1. Drop rows where the target is missing
        data = data.dropna(subset=["burnout_level"]).copy()

        # 2. Split
        X = data.drop("burnout_level", axis=1)
        y = data["burnout_level"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.2,
            random_state=42,
            stratify=y
        )

        # 3. Preprocessing pipeline
        num_cols = X_train.select_dtypes(include='number').columns

        num_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler())
        ])

        # 4. Fit only on train
        X_train[num_cols] = num_pipeline.fit_transform(X_train[num_cols])
        X_test[num_cols] = num_pipeline.transform(X_test[num_cols])

        # 5. Encode target
        le = LabelEncoder()
        y_train = le.fit_transform(y_train)
        y_test = le.transform(y_test)

        # 6. Combine back
        train_df = X_train.copy()
        train_df["burnout_level"] = y_train

        test_df = X_test.copy()
        test_df["burnout_level"] = y_test

        # 7. Save artifacts
        joblib.dump(num_pipeline, f"{self.config.preprocessor_path}/preprocessor.pkl")
        joblib.dump(le, f"{self.config.preprocessor_path}/label_encoder.pkl")

        train_df.to_csv(f"{self.config.root_dir}/train.csv", index=False)
        test_df.to_csv(f"{self.config.root_dir}/test.csv", index=False)

In [15]:
configurationManager = ConfigurationManager()
data_transformation_config = configurationManager.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.transform_and_save()

[2026-04-16 20:27:00,667: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-16 20:27:00,668: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-16 20:27:00,670: INFO: common: created directory at: artifacts]
[2026-04-16 20:27:00,671: INFO: common: created directory at: artifacts/data_transformation]
[2026-04-16 20:27:00,671: INFO: common: created directory at: artifacts/data_transformation/preprocessor]
[2026-04-16 20:27:00,672: INFO: common: created directory at: artifacts/data_transformation]
